# Universal Loader Validation

End-to-end test of LakeForge's format-agnostic ingestion: generates sample data in every supported format (csv, tsv, json, jsonl, parquet, avro, orc, xml, xlsx, delta, fixed-width text) and loads each through `UniversalLoader`, then exercises config-driven loading and mixed-directory loading.

In [0]:
%pip install openpyxl chardet


In [0]:
# Setup & generate sample files in every supported format
import sys, json, os
sys.path.append(".")

import pandas as pd
from pyspark.sql import Row
from lakeforge.ingestion import create_universal_loader

base = "/Volumes/lakehouse/raw/lakeforge_data/test_formats"
os.makedirs(base, exist_ok=True)

sample = spark.createDataFrame([
    Row(id=1, name="alice", country="US"),
    Row(id=2, name="bob", country="IN"),
    Row(id=3, name="carol", country="UK"),
])
sample.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{base}/csv")
sample.coalesce(1).write.mode("overwrite").option("header", "true").option("delimiter", "\t").csv(f"{base}/tsv")
sample.coalesce(1).write.mode("overwrite").json(f"{base}/json")
sample.coalesce(1).write.mode("overwrite").parquet(f"{base}/parquet")
sample.coalesce(1).write.mode("overwrite").format("avro").save(f"{base}/avro")
sample.coalesce(1).write.mode("overwrite").format("orc").save(f"{base}/orc")
sample.coalesce(1).write.mode("overwrite").format("xml").option("rootTag", "records").option("rowTag", "record").save(f"{base}/xml")
sample.coalesce(1).write.mode("overwrite").format("delta").save(f"{base}/delta")

with open(f"{base}/data.jsonl", "w") as f:
    for r in [1, 2, 3]:
        f.write(json.dumps({"id": r, "name": f"user{r}", "country": "XX"}) + "\n")

with open(f"{base}/fixed.txt", "w") as f:
    f.write("id  name  country\n")
    f.write("1   alice US     \n")
    f.write("2   bob   IN     \n")
    f.write("3   carol UK     \n")

# Write xlsx locally first (zipfile seeks are unreliable directly on volume mounts), then copy
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    tmp_xlsx = f"{tmp}/data.xlsx"
    pd.DataFrame({"id": [1, 2, 3], "name": ["alice", "bob", "carol"], "country": ["US", "IN", "UK"]}).to_excel(tmp_xlsx, index=False)
    import shutil
    shutil.copyfile(tmp_xlsx, f"{base}/data.xlsx")

print("Sample files written:", sorted(os.listdir(base)))

In [0]:
# Validate: load every format through the UniversalLoader
# Reload framework modules so edits made during this session are picked up
import importlib
import lakeforge.ingestion.files.xml_loader as _xml_mod
import lakeforge.ingestion.files.universal_loader as _uni_mod
importlib.reload(_xml_mod)
importlib.reload(_uni_mod)
from lakeforge.ingestion.files.universal_loader import UniversalLoader

loader = UniversalLoader(spark)

# Debug: confirm XML row-tag auto-detection on a directory of part files
print("XML row tag detected:", _xml_mod.XMLLoader(spark).detect_row_tag(f"{base}/xml"))

cases = [
    # Directories written by Spark have extension-less part files -> pass file_format explicitly
    ("csv",          f"{base}/csv",        {"file_format": "csv"}),
    ("tsv",          f"{base}/tsv",        {"file_format": "tsv"}),
    ("json",         f"{base}/json",       {"file_format": "json"}),
    ("parquet",      f"{base}/parquet",    {"file_format": "parquet"}),
    ("avro",         f"{base}/avro",       {"file_format": "avro"}),
    ("orc",          f"{base}/orc",        {"file_format": "orc"}),
    ("xml",          f"{base}/xml",        {"file_format": "xml"}),
    ("delta",        f"{base}/delta",      {"file_format": "delta"}),
    # Single files with real extensions -> auto-detection path
    ("jsonl",        f"{base}/data.jsonl", {}),
    ("excel",        f"{base}/data.xlsx",   {}),
    ("fixed-width",  f"{base}/fixed.txt",  {"column_specs": {"id": (0, 3), "name": (3, 9), "country": (9, 16)}}),
]

print(f"{'format':<13}{'rows':>5}  columns")
failures = []
for label, path, opts in cases:
    try:
        df = loader.load(path, **opts)
        print(f"{label:<13}{df.count():>5}  {df.columns}")
    except Exception as e:
        failures.append((label, str(e)[:300]))
        print(f"{label:<13}FAILED: {str(e)[:300]}")

assert not failures, f"Format load failures: {failures}"
print("\n✅ ALL FORMATS LOADED SUCCESSFULLY")

In [0]:
# Validate config-driven loading + format auto-detection + mixed directories
import shutil
from lakeforge.metadata.config_parser import IngestionConfig

# 1) Config with explicit format + reader options passthrough (rowTag for XML)
xml_cfg = IngestionConfig(
    source_name="customers_xml", source_type="xml", file_format="xml",
    source_path=f"{base}/xml",
    target_table="customers", target_catalog="lakehouse", target_schema="bronze",
    reader_options={"rowTag": "record"},
)
df_xml = loader.load_from_config(xml_cfg)
print(f"load_from_config(xml): {df_xml.count()} rows, columns={df_xml.columns}")
assert df_xml.count() == 3

# 2) Config relying on extension auto-detection (jsonl)
jsonl_cfg = IngestionConfig(
    source_name="events", source_type="jsonl", file_format="jsonl",
    source_path=f"{base}/data.jsonl",
    target_table="events", target_catalog="lakehouse", target_schema="bronze",
)
df_jsonl = loader.load_from_config(jsonl_cfg)
print(f"load_from_config(jsonl): {df_jsonl.count()} rows, columns={df_jsonl.columns}")
assert df_jsonl.count() == 3

# 3) Mixed directory (single files of different formats side by side)
mixed_dir = f"{base}/mixed"
os.makedirs(mixed_dir, exist_ok=True)
shutil.copyfile(f"{base}/data.jsonl", f"{mixed_dir}/events.jsonl")
shutil.copyfile(f"{base}/data.xlsx", f"{mixed_dir}/accounts.xlsx")
shutil.copyfile(f"{base}/fixed.txt", f"{mixed_dir}/positions.dat")
with open(f"{mixed_dir}/customers.csv", "w") as f:
    f.write("id,name,country\n1,alice,US\n2,bob,IN\n3,carol,UK\n")

mixed = loader.load_mixed_directory(mixed_dir)
for fmt, df in mixed.items():
    print(f"mixed dir -> {fmt}: {df.count()} rows")
assert set(mixed.keys()) == {"json", "excel", "text", "csv"}

print("\n✅ CONFIG-DRIVEN, AUTO-DETECT, AND MIXED-DIRECTORY LOADING ALL PASSED")

In [0]:
# Validate multi-source config file resolution (usecase-style single file with many sources)
import importlib
import lakeforge.metadata.config_parser as _cp
importlib.reload(_cp)

_cfg_dir = "configs"

# Bronze ingestion: per-source storage account, catalog, bronze schema, silver table
for _name in ["bad_duplicate_keys", "customers", "transactions"]:
    _c = _cp.ConfigParser.get_ingestion_config(_cfg_dir, _name, "dev")
    print(f"{_name}: storage={_c.storage_account} {_c.target_catalog}.{_c.target_schema}.{_c.target_table} -> silver={_c.silver_schema}.{_c.silver_table} mode={_c.mode}")

# DQ config follows the per-source catalog/schema
_d = _cp.ConfigParser.get_dq_config(_cfg_dir, "bad_duplicate_keys", "dev")
print(f"DQ: {_d.catalog}.{_d.schema}.{_d.table_name} rules={len(_d.rules)}")

# Silver transformation configs resolve, with the column schema living at silver level
for _t in ["bad_duplicate_keys_clean", "customers_clean"]:
    _s = _cp.ConfigParser.get_silver_transformation_config(_cfg_dir, _t, "dev")
    print(f"silver {_t}: source={_s['source_table']} schema_cols={len(_s['schema'])} steps={len(_s['transformations'])}")

# Confirm the package still imports cleanly after drift removal
import lakeforge
assert not hasattr(lakeforge, "SchemaDriftDetector"), "drift export should be gone"
print("\n✅ Multi-source configs resolve; lakeforge imports clean without drift")